# IHP sg13g2 — 5T OTA layout from foundry PyCells (Python 3.11)

Generate a rough layout of the `analog-db` **amp_001_5t** 5T-OTA from IHP foundry PyCells,
write GDS, render it, and run **DRC + LVS** through the PDK decks.

Needs: `PDK_ROOT` → dir containing `ihp-sg13g2`, the pip `klayout` module, and (for signoff)
a working `klayout` binary on `PATH`. Run with a Python 3.11 that has `klayout` (e.g. the
`ai_env` conda env). See `../README.md`.

## 1. Bootstrap the PDK PyCells and instantiate one device
The foundry PyCells only fail inside KLayout's embedded Python 3.6; here (3.11) they import fine.

In [ ]:
import pdk as P
pya = P.bootstrap()
print('libraries:', pya.Library.library_names())
ly = pya.Layout(); ly.dbu = 0.001
nmos = ly.create_cell('nmos', 'SG13_dev', {'w': '5u', 'l': '0.5u', 'ng': '1'})
print('nmos PCell bbox (um):', nmos.dbbox().to_s())

## 2. Build the 5T OTA and write GDS

In [ ]:
import os, gen_5t_ota
layout, top = gen_5t_ota.build()
gds = os.path.abspath('ota_5t.gds')
layout.write(gds)
print('wrote', gds, '— bbox um:', top.dbbox().to_s())

## 3. Render the layout (headless, via `klayout.lay`)

In [ ]:
import klayout.lay as lay
from IPython.display import Image
lv = lay.LayoutView()
lv.load_layout(gds, 0)
try: lv.load_layer_props(os.path.join(P.KL, 'tech/sg13g2.lyp'))
except Exception as e: print('lyp:', e)
lv.max_hier(); lv.zoom_fit()
lv.save_image('ota_5t.png', 1800, 700)
Image('ota_5t.png')

## 4. Signoff — DRC + LVS
DRC runs with `--no_density` (geometric clean; fill/min-density deferred to full-chip fill).
LVS compares against the flat reference netlist `ota_5t_lvs.spice`.

In [ ]:
import signoff
run = os.path.abspath('signoff_out'); os.makedirs(run, exist_ok=True)
drc_ok, _ = signoff.run_drc(gds, 'ota_5t', os.path.join(run, 'drc'))
lvs_ok, lvs_out = signoff.run_lvs(gds, os.path.abspath('ota_5t_lvs.spice'), 'ota_5t', os.path.join(run, 'lvs'))
print('DRC (--no_density):', 'PASS' if drc_ok else 'FAIL')
print('LVS:               ', 'PASS' if lvs_ok else 'FAIL')
print([l for l in lvs_out.splitlines() if 'match' in l.lower()][-1:])